# Boundary-aware cell tracking

This notebook replaces one-frame-only association with persistent track states
for cells touching the volume boundary.

Key changes:

- physical-space distance calculation using the voxel spacing;
- explicit boundary-face metadata for every detection;
- boundary-specific assignment weights that ignore unreliable truncated shape;
- relaxed boundary volume gating;
- short-term boundary-track memory and reacquisition before creating a new ID;
- reliable feature templates updated only from fully visible observations;
- separate event and prediction files for auditing boundary behavior.


In [12]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist


In [13]:
# ------------------------------------------------------------
# Paths and sample configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

PROCESSED_DIR = (
    DATA_ROOT
    / "processed"
    / "stage_6_processed_dataset"
    / SAMPLE_ID
)

FEATURE_DIR = PROCESSED_DIR / "cells"

OUTPUT_DIR = (
    DATA_ROOT
    / "processed"
    / "stage_7_cell_tracking"
)

# Volume geometry for this dataset, in (Z, Y, X).
VOLUME_SHAPE_ZYX = np.asarray(
    [64, 256, 256],
    dtype=int,
)

VOXEL_SIZE_ZYX = np.asarray(
    [1.625, 0.40625, 0.40625],
    dtype=float,
)


In [14]:
# ------------------------------------------------------------
# Load per-frame cell detections
# ------------------------------------------------------------

cell_files = sorted(FEATURE_DIR.glob("t*.csv"))

if not cell_files:
    raise FileNotFoundError(
        f"No per-frame cell CSV files were found in:\n{FEATURE_DIR}"
    )

time_frames = [
    pd.read_csv(file)
    for file in cell_files
]

print(f"Loaded {len(time_frames)} timepoints.")


Loaded 20 timepoints.


In [15]:
time_frames[0].head()

,cell_id,volume_voxels,z_min,y_min,x_min,z_max,y_max,x_max,centroid_z,centroid_y,...,equivalent_radius,axis_major,axis_middle,axis_minor,elongation,flatness,anisotropy,convex_volume,solidity,compactness
0,1,186.0,0,0,46,2,14,60,0.231183,6.354839,...,3.541127,3.312989,2.928501,0.420773,1.131292,6.959811,7.873577,77.166667,2.410367,0.023965
1,2,222.0,0,0,65,2,16,80,0.153153,6.815315,...,3.756253,4.082647,3.323817,0.354509,1.228301,9.375832,11.516344,88.000000,2.522727,0.021110
2,3,240.0,0,18,59,4,35,72,0.320833,26.012500,...,3.855146,3.955908,2.965969,0.548120,1.333766,5.411165,7.217226,171.666667,1.398058,0.009131
3,4,737.0,0,55,48,4,72,68,1.074627,62.743555,...,5.603503,4.670576,3.679390,0.964733,1.269389,3.813895,4.841316,520.500000,1.415946,0.014695
4,5,343.0,0,89,28,2,109,43,0.364431,98.571429,...,4.342453,4.462224,3.304942,0.480760,1.350167,6.874411,9.281601,154.833333,2.215285,0.023338


In [16]:
for t, df in enumerate(time_frames):
    print(f"t={t:03d}: {len(df)} cells")


t=000: 202 cells
t=001: 211 cells
t=002: 209 cells
t=003: 214 cells
t=004: 211 cells
t=005: 199 cells
t=006: 207 cells
t=007: 209 cells
t=008: 205 cells
t=009: 206 cells
t=010: 209 cells
t=011: 208 cells
t=012: 215 cells
t=013: 220 cells
t=014: 222 cells
t=015: 222 cells
t=016: 206 cells
t=017: 211 cells
t=018: 213 cells
t=019: 224 cells


In [17]:
# ============================================================
# Boundary-aware tracking configuration
# ============================================================

# All assignment distances are now measured in physical units.
MAX_DISTANCE_UM = 6.0
BOUNDARY_MAX_DISTANCE_UM = 10.0
GLOBAL_SHIFT_MAX_PAIR_DISTANCE_UM = 12.0
BOUNDARY_DISTANCE_PER_MISSING_FRAME_UM = 2.0

# Interior cells retain the strict morphology gate.
MAX_VOLUME_RATIO_INTERIOR = 1.5

# Boundary-truncated cells may change measured volume considerably.
MAX_VOLUME_RATIO_BOUNDARY = 4.0

# A boundary track may remain unmatched for this many frames and
# still be considered for reacquisition.
BOUNDARY_MAX_MISSING_FRAMES = 2

# Objects whose bounding boxes lie within this physical margin of
# a volume face are considered boundary affected.
BOUNDARY_MARGIN_UM = 2.0

# Interior assignment weights.
INTERIOR_W_DISTANCE = 0.40
INTERIOR_W_SIZE = 0.20
INTERIOR_W_SHAPE = 0.25
INTERIOR_W_INTENSITY = 0.10
INTERIOR_W_BBOX = 0.05

# Boundary assignment weights. Truncated size, shape, and bounding-box
# measurements are deliberately excluded.
BOUNDARY_W_DISTANCE = 0.70
BOUNDARY_W_MOTION = 0.15
BOUNDARY_W_INTENSITY = 0.10
BOUNDARY_W_FACE = 0.05

# Velocity/template update parameters.
VELOCITY_EMA_ALPHA = 0.60
TEMPLATE_EMA_ALPHA = 0.20

INVALID_COST = 1e6
EPS = 1e-8

print(
    "Boundary margin in voxels:",
    np.ceil(
        BOUNDARY_MARGIN_UM / VOXEL_SIZE_ZYX
    ).astype(int),
)


Boundary margin in voxels: [2 5 5]


In [18]:
# ============================================================
# Boundary metadata and assignment helpers
# ============================================================

SIZE_FEATURES = [
    "volume_voxels",
    "extent",
    "equivalent_radius",
]

SHAPE_FEATURES = [
    "elongation",
    "flatness",
    "anisotropy",
    "solidity",
    "compactness",
]

INTENSITY_FEATURES = [
    "intensity_mean",
    "intensity_std",
    "intensity_cv",
]

BBOX_FEATURES = [
    "bbox_depth",
    "bbox_height",
    "bbox_width",
]

TEMPLATE_FEATURES = sorted(
    set(
        SIZE_FEATURES
        + SHAPE_FEATURES
        + INTENSITY_FEATURES
        + BBOX_FEATURES
    )
)


def annotate_boundary_metadata(
    detections: pd.DataFrame,
) -> pd.DataFrame:
    """Add boundary-contact information to one frame's detections."""

    required_bbox_columns = {
        "z_min", "z_max",
        "y_min", "y_max",
        "x_min", "x_max",
    }

    missing = required_bbox_columns.difference(
        detections.columns
    )

    if missing:
        raise KeyError(
            "Boundary-aware tracking requires bounding-box columns. "
            f"Missing: {sorted(missing)}"
        )

    result = detections.copy()

    margin_zyx = np.ceil(
        BOUNDARY_MARGIN_UM / VOXEL_SIZE_ZYX
    ).astype(int)

    z_size, y_size, x_size = VOLUME_SHAPE_ZYX
    z_margin, y_margin, x_margin = margin_zyx

    # The stored bbox maxima are treated as upper/exclusive bounds.
    result["touches_z_min"] = result["z_min"] <= z_margin
    result["touches_z_max"] = result["z_max"] >= (
        z_size - z_margin
    )

    result["touches_y_min"] = result["y_min"] <= y_margin
    result["touches_y_max"] = result["y_max"] >= (
        y_size - y_margin
    )

    result["touches_x_min"] = result["x_min"] <= x_margin
    result["touches_x_max"] = result["x_max"] >= (
        x_size - x_margin
    )

    face_columns = [
        "touches_z_min",
        "touches_z_max",
        "touches_y_min",
        "touches_y_max",
        "touches_x_min",
        "touches_x_max",
    ]

    result["touches_boundary"] = (
        result[face_columns].any(axis=1)
    )

    result["boundary_face_count"] = (
        result[face_columns].sum(axis=1).astype(int)
    )

    face_names = [
        "z_min", "z_max",
        "y_min", "y_max",
        "x_min", "x_max",
    ]

    result["boundary_faces"] = [
        "|".join(
            face_name
            for face_name, flag in zip(
                face_names,
                flags,
            )
            if bool(flag)
        )
        for flags in result[face_columns].to_numpy()
    ]

    centroids = result[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    lower_distance_um = (
        centroids * VOXEL_SIZE_ZYX
    )

    upper_distance_um = (
        (
            VOLUME_SHAPE_ZYX
            - 1
            - centroids
        )
        * VOXEL_SIZE_ZYX
    )

    result["distance_to_boundary_um"] = np.min(
        np.concatenate(
            [lower_distance_um, upper_distance_um],
            axis=1,
        ),
        axis=1,
    )

    result["is_boundary_partial"] = (
        result["touches_boundary"]
    )

    return result


def parse_boundary_faces(value) -> set[str]:
    """Convert the stored pipe-separated face string into a set."""

    if value is None or pd.isna(value):
        return set()

    text = str(value).strip()

    if not text:
        return set()

    return {
        face
        for face in text.split("|")
        if face
    }


def physical_coordinates(
    detections: pd.DataFrame,
) -> np.ndarray:
    """Return centroid coordinates in physical (Z, Y, X) units."""

    coords_voxel = detections[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    return coords_voxel * VOXEL_SIZE_ZYX


def robust_global_shift_physical(
    previous_positions: np.ndarray,
    current_positions: np.ndarray,
) -> np.ndarray:
    """Estimate global displacement from mutual nearest-neighbour pairs.

    This is less sensitive than subtracting frame medians when cells enter,
    leave, or temporarily disappear at a boundary.
    """

    if (
        len(previous_positions) == 0
        or len(current_positions) == 0
    ):
        return np.zeros(3, dtype=float)

    distances = cdist(
        previous_positions,
        current_positions,
    )

    nearest_current = np.argmin(
        distances,
        axis=1,
    )

    nearest_previous = np.argmin(
        distances,
        axis=0,
    )

    matched_displacements = []

    for previous_index, current_index in enumerate(
        nearest_current
    ):
        if (
            nearest_previous[current_index]
            != previous_index
        ):
            continue

        if (
            distances[
                previous_index,
                current_index,
            ]
            > GLOBAL_SHIFT_MAX_PAIR_DISTANCE_UM
        ):
            continue

        matched_displacements.append(
            current_positions[current_index]
            - previous_positions[previous_index]
        )

    if matched_displacements:
        return np.median(
            np.asarray(
                matched_displacements,
                dtype=float,
            ),
            axis=0,
        )

    # Conservative fallback when no mutual pair survives.
    return np.zeros(3, dtype=float)


def make_feature_template(
    detection: pd.Series,
) -> dict[str, float]:
    """Create a numeric feature template from one detection."""

    template = {}

    for feature in TEMPLATE_FEATURES:
        if feature not in detection.index:
            continue

        value = detection[feature]

        if pd.notna(value):
            template[feature] = float(value)

    return template


def update_feature_template(
    state: dict,
    detection: pd.Series,
) -> None:
    """Update a reliable template using only fully visible detections."""

    if bool(detection["touches_boundary"]):
        return

    values = make_feature_template(detection)

    if not state["template_reliable"]:
        state["template"] = values
        state["template_reliable"] = True
        state["template_count"] = 1
        return

    for feature, value in values.items():
        old_value = state["template"].get(
            feature,
            value,
        )

        state["template"][feature] = (
            (1.0 - TEMPLATE_EMA_ALPHA)
            * old_value
            + TEMPLATE_EMA_ALPHA
            * value
        )

    state["template_count"] += 1


def make_track_state(
    *,
    track_id: int,
    frame: int,
    detection: pd.Series,
) -> dict:
    """Create the persistent state for one new track."""

    position_voxel = detection[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    position_physical = (
        position_voxel * VOXEL_SIZE_ZYX
    )

    is_boundary = bool(
        detection["touches_boundary"]
    )

    return {
        "track_id": int(track_id),
        "active": True,
        "last_frame": int(frame),
        "last_position_voxel": position_voxel,
        "last_position_physical": position_physical,
        "previous_position_physical": None,
        "velocity_physical": np.zeros(3, dtype=float),
        "velocity_valid": False,
        "last_detection": detection.to_dict(),
        "missed_frames": 0,
        "boundary_pending": is_boundary,
        "last_boundary_faces": parse_boundary_faces(
            detection["boundary_faces"]
        ),
        "template": make_feature_template(detection),
        "template_reliable": not is_boundary,
        "template_count": 1 if not is_boundary else 0,
    }


def state_reference_value(
    state: dict,
    feature: str,
) -> float:
    """Read a feature from the reliable template or last observation."""

    if (
        state["template_reliable"]
        and feature in state["template"]
    ):
        return float(state["template"][feature])

    value = state["last_detection"].get(
        feature,
        np.nan,
    )

    return float(value) if pd.notna(value) else np.nan


def predict_state_position(
    state: dict,
    target_frame: int,
    global_shift_physical: np.ndarray,
) -> np.ndarray:
    """Predict a track position in physical coordinates."""

    frame_gap = int(target_frame) - int(
        state["last_frame"]
    )

    if frame_gap <= 0:
        return state["last_position_physical"].copy()

    if state["velocity_valid"]:
        displacement = (
            state["velocity_physical"] * frame_gap
        )
    else:
        displacement = (
            global_shift_physical * frame_gap
        )

    return (
        state["last_position_physical"]
        + displacement
    )


def normalized_pairwise_cost(
    previous_values: np.ndarray,
    current_values: np.ndarray,
) -> np.ndarray:
    """Relative absolute difference with robust NaN handling."""

    previous_values = np.asarray(
        previous_values,
        dtype=float,
    )

    current_values = np.asarray(
        current_values,
        dtype=float,
    )

    cost = (
        np.abs(
            previous_values[:, None]
            - current_values[None, :]
        )
        / (
            np.maximum(
                np.abs(previous_values[:, None]),
                np.abs(current_values[None, :]),
            )
            + EPS
        )
    )

    return np.nan_to_num(
        cost,
        nan=1.0,
        posinf=1.0,
        neginf=1.0,
    )


def feature_group_cost(
    states: list[dict],
    detections: pd.DataFrame,
    feature_names: list[str],
) -> np.ndarray:
    """Average relative cost over available features in one group."""

    costs = []

    for feature in feature_names:
        if feature not in detections.columns:
            continue

        previous_values = np.asarray(
            [
                state_reference_value(
                    state,
                    feature,
                )
                for state in states
            ],
            dtype=float,
        )

        current_values = detections[
            feature
        ].to_numpy(dtype=float)

        costs.append(
            normalized_pairwise_cost(
                previous_values,
                current_values,
            )
        )

    if not costs:
        return np.zeros(
            (len(states), len(detections)),
            dtype=float,
        )

    return np.mean(
        np.stack(costs, axis=0),
        axis=0,
    )


def motion_consistency_cost(
    states: list[dict],
    current_positions: np.ndarray,
) -> np.ndarray:
    """Compare candidate motion direction with each track's velocity."""

    result = np.zeros(
        (len(states), len(current_positions)),
        dtype=float,
    )

    for state_index, state in enumerate(states):
        if not state["velocity_valid"]:
            continue

        expected = state["velocity_physical"]
        expected_norm = np.linalg.norm(expected)

        if expected_norm <= EPS:
            continue

        displacements = (
            current_positions
            - state["last_position_physical"][None, :]
        )

        displacement_norms = np.linalg.norm(
            displacements,
            axis=1,
        )

        valid = displacement_norms > EPS

        cosine = np.ones(
            len(current_positions),
            dtype=float,
        )

        cosine[valid] = (
            displacements[valid] @ expected
            / (
                displacement_norms[valid]
                * expected_norm
            )
        )

        cosine = np.clip(cosine, -1.0, 1.0)

        # 0 means aligned, 1 means opposite.
        result[state_index] = (
            1.0 - cosine
        ) / 2.0

    return result


def boundary_face_cost(
    states: list[dict],
    detections: pd.DataFrame,
) -> np.ndarray:
    """Penalize jumps between incompatible volume faces."""

    current_faces = [
        parse_boundary_faces(value)
        for value in detections["boundary_faces"]
    ]

    current_boundary = detections[
        "touches_boundary"
    ].to_numpy(dtype=bool)

    result = np.zeros(
        (len(states), len(detections)),
        dtype=float,
    )

    for state_index, state in enumerate(states):
        previous_faces = state[
            "last_boundary_faces"
        ]

        previous_boundary = bool(
            state["last_detection"].get(
                "touches_boundary",
                False,
            )
        ) or state["boundary_pending"]

        if not previous_boundary:
            continue

        for detection_index, faces in enumerate(
            current_faces
        ):
            # Moving from a boundary into the interior is valid.
            if not current_boundary[detection_index]:
                continue

            if previous_faces and faces:
                if previous_faces.isdisjoint(faces):
                    result[
                        state_index,
                        detection_index,
                    ] = 1.0

    return result


def assign_track_states(
    *,
    states: list[dict],
    detections: pd.DataFrame,
    current_frame: int,
    global_shift_physical: np.ndarray,
):
    """Assign active/pending tracks to current detections."""

    if not states or detections.empty:
        return {
            "rows": np.array([], dtype=int),
            "cols": np.array([], dtype=int),
            "distance_matrix": np.empty(
                (len(states), len(detections))
            ),
            "cost_matrix": np.empty(
                (len(states), len(detections))
            ),
            "predicted_positions": np.empty(
                (len(states), 3)
            ),
            "boundary_related": np.empty(
                (len(states), len(detections)),
                dtype=bool,
            ),
        }

    predicted_positions = np.vstack(
        [
            predict_state_position(
                state,
                current_frame,
                global_shift_physical,
            )
            for state in states
        ]
    )

    current_positions = physical_coordinates(
        detections
    )

    distance_matrix = cdist(
        predicted_positions,
        current_positions,
    )

    frame_gaps = np.asarray(
        [
            current_frame - state["last_frame"]
            for state in states
        ],
        dtype=int,
    )

    previous_boundary = np.asarray(
        [
            bool(
                state["last_detection"].get(
                    "touches_boundary",
                    False,
                )
            )
            or state["boundary_pending"]
            or state["missed_frames"] > 0
            for state in states
        ],
        dtype=bool,
    )

    current_boundary = detections[
        "touches_boundary"
    ].to_numpy(dtype=bool)

    boundary_related = (
        previous_boundary[:, None]
        | current_boundary[None, :]
    )

    distance_limit = np.where(
        boundary_related,
        (
            BOUNDARY_MAX_DISTANCE_UM
            + BOUNDARY_DISTANCE_PER_MISSING_FRAME_UM
            * np.maximum(
                frame_gaps[:, None] - 1,
                0,
            )
        ),
        MAX_DISTANCE_UM,
    )

    distance_cost = (
        distance_matrix
        / np.maximum(distance_limit, EPS)
    )

    size_cost = feature_group_cost(
        states,
        detections,
        SIZE_FEATURES,
    )

    shape_cost = feature_group_cost(
        states,
        detections,
        SHAPE_FEATURES,
    )

    intensity_cost = feature_group_cost(
        states,
        detections,
        INTENSITY_FEATURES,
    )

    bbox_cost = feature_group_cost(
        states,
        detections,
        BBOX_FEATURES,
    )

    motion_cost = motion_consistency_cost(
        states,
        current_positions,
    )

    face_cost = boundary_face_cost(
        states,
        detections,
    )

    interior_cost = (
        INTERIOR_W_DISTANCE * distance_cost
        + INTERIOR_W_SIZE * size_cost
        + INTERIOR_W_SHAPE * shape_cost
        + INTERIOR_W_INTENSITY * intensity_cost
        + INTERIOR_W_BBOX * bbox_cost
    )

    boundary_cost = (
        BOUNDARY_W_DISTANCE * distance_cost
        + BOUNDARY_W_MOTION * motion_cost
        + BOUNDARY_W_INTENSITY * intensity_cost
        + BOUNDARY_W_FACE * face_cost
    )

    cost_matrix = np.where(
        boundary_related,
        boundary_cost,
        interior_cost,
    )

    previous_volume = np.asarray(
        [
            state_reference_value(
                state,
                "volume_voxels",
            )
            for state in states
        ],
        dtype=float,
    )

    current_volume = detections[
        "volume_voxels"
    ].to_numpy(dtype=float)

    volume_ratio = (
        np.maximum(
            previous_volume[:, None],
            current_volume[None, :],
        )
        / (
            np.minimum(
                previous_volume[:, None],
                current_volume[None, :],
            )
            + EPS
        )
    )

    maximum_volume_ratio = np.where(
        boundary_related,
        MAX_VOLUME_RATIO_BOUNDARY,
        MAX_VOLUME_RATIO_INTERIOR,
    )

    invalid = (
        (distance_matrix > distance_limit)
        | (volume_ratio > maximum_volume_ratio)
    )

    cost_matrix = cost_matrix.copy()
    cost_matrix[invalid] = INVALID_COST

    rows, cols = linear_sum_assignment(
        cost_matrix
    )

    valid = (
        cost_matrix[rows, cols] < INVALID_COST
    )

    return {
        "rows": rows[valid],
        "cols": cols[valid],
        "distance_matrix": distance_matrix,
        "cost_matrix": cost_matrix,
        "predicted_positions": predicted_positions,
        "boundary_related": boundary_related,
    }


def update_matched_state(
    *,
    state: dict,
    detection: pd.Series,
    current_frame: int,
) -> tuple[bool, bool]:
    """Update a matched track and return transition flags."""

    previous_boundary = bool(
        state["last_detection"].get(
            "touches_boundary",
            False,
        )
    )

    was_reacquired = (
        state["missed_frames"] > 0
    )

    new_position_voxel = detection[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    new_position_physical = (
        new_position_voxel * VOXEL_SIZE_ZYX
    )

    frame_gap = (
        current_frame - state["last_frame"]
    )

    observed_velocity = (
        (
            new_position_physical
            - state["last_position_physical"]
        )
        / max(frame_gap, 1)
    )

    if state["velocity_valid"]:
        state["velocity_physical"] = (
            (1.0 - VELOCITY_EMA_ALPHA)
            * state["velocity_physical"]
            + VELOCITY_EMA_ALPHA
            * observed_velocity
        )
    else:
        state["velocity_physical"] = (
            observed_velocity
        )
        state["velocity_valid"] = True

    state["previous_position_physical"] = (
        state["last_position_physical"].copy()
    )

    state["last_position_voxel"] = (
        new_position_voxel
    )

    state["last_position_physical"] = (
        new_position_physical
    )

    state["last_frame"] = int(current_frame)
    state["last_detection"] = detection.to_dict()
    state["missed_frames"] = 0

    current_boundary = bool(
        detection["touches_boundary"]
    )

    state["boundary_pending"] = (
        current_boundary
    )

    state["last_boundary_faces"] = (
        parse_boundary_faces(
            detection["boundary_faces"]
        )
    )

    update_feature_template(
        state,
        detection,
    )

    return was_reacquired, previous_boundary


In [19]:
# ============================================================
# Annotate detections with boundary metadata
# ============================================================

time_frames = [
    annotate_boundary_metadata(frame)
    for frame in time_frames
]

boundary_counts = pd.DataFrame(
    {
        "frame": np.arange(len(time_frames)),
        "detections": [
            len(frame)
            for frame in time_frames
        ],
        "boundary_detections": [
            int(frame["touches_boundary"].sum())
            for frame in time_frames
        ],
    }
)

boundary_counts.head()


,frame,detections,boundary_detections
0,0,202,74
1,1,211,77
2,2,209,75
3,3,214,76
4,4,211,72


In [20]:
# ============================================================
# Boundary-aware tracking with short-term reacquisition
# ============================================================

track_states: dict[int, dict] = {}
track_records: list[dict] = []
boundary_events: list[dict] = []
boundary_predictions: list[dict] = []

next_track_id = 0


def append_track_record(
    *,
    track_id: int,
    frame: int,
    cell_index: int,
    detection: pd.Series,
    match_type: str,
    match_distance_um: float | None,
    match_cost: float | None,
    template_reliable: bool,
) -> None:
    """Append one observed detection to the output track table."""

    track_records.append(
        {
            "track_id": int(track_id),
            "frame": int(frame),
            "cell": int(cell_index),
            "cell_id": int(
                detection.get(
                    "cell_id",
                    cell_index,
                )
            ),
            "z": float(detection["centroid_z"]),
            "y": float(detection["centroid_y"]),
            "x": float(detection["centroid_x"]),
            "volume": float(
                detection["volume_voxels"]
            ),
            "touches_boundary": bool(
                detection["touches_boundary"]
            ),
            "boundary_faces": str(
                detection["boundary_faces"]
            ),
            "distance_to_boundary_um": float(
                detection[
                    "distance_to_boundary_um"
                ]
            ),
            "boundary_state": (
                "ACTIVE_BOUNDARY"
                if bool(
                    detection["touches_boundary"]
                )
                else "ACTIVE_INTERIOR"
            ),
            "match_type": str(match_type),
            "match_distance_um": (
                np.nan
                if match_distance_um is None
                else float(match_distance_um)
            ),
            "match_cost": (
                np.nan
                if match_cost is None
                else float(match_cost)
            ),
            "template_reliable": bool(
                template_reliable
            ),
        }
    )


# ------------------------------------------------------------
# Initialize tracks from the first frame
# ------------------------------------------------------------

first_frame = time_frames[0]

for cell_index, detection in first_frame.iterrows():
    track_id = next_track_id
    next_track_id += 1

    state = make_track_state(
        track_id=track_id,
        frame=0,
        detection=detection,
    )

    track_states[track_id] = state

    match_type = (
        "boundary_entry"
        if bool(detection["touches_boundary"])
        else "initial"
    )

    append_track_record(
        track_id=track_id,
        frame=0,
        cell_index=int(cell_index),
        detection=detection,
        match_type=match_type,
        match_distance_um=None,
        match_cost=None,
        template_reliable=state[
            "template_reliable"
        ],
    )

    if bool(detection["touches_boundary"]):
        boundary_events.append(
            {
                "track_id": track_id,
                "frame": 0,
                "event_type": "boundary_entry",
                "boundary_faces": detection[
                    "boundary_faces"
                ],
                "missing_frames": 0,
                "reacquired_frame": np.nan,
                "confidence": np.nan,
            }
        )


# ------------------------------------------------------------
# Process each subsequent frame
# ------------------------------------------------------------

for current_frame in range(
    1,
    len(time_frames),
):
    detections = time_frames[current_frame]

    eligible_states = []

    for state in track_states.values():
        if not state["active"]:
            continue

        frame_gap = (
            current_frame
            - state["last_frame"]
        )

        # All tracks are eligible for the immediate next frame.
        if frame_gap == 1:
            eligible_states.append(state)
            continue

        # Only boundary-pending tracks survive a longer gap.
        if (
            state["boundary_pending"]
            and frame_gap
            <= BOUNDARY_MAX_MISSING_FRAMES + 1
        ):
            eligible_states.append(state)
            continue

        state["active"] = False

    previous_frame_positions = np.asarray(
        [
            state["last_position_physical"]
            for state in eligible_states
            if state["last_frame"]
            == current_frame - 1
        ],
        dtype=float,
    ).reshape(-1, 3)

    current_positions = physical_coordinates(
        detections
    )

    global_shift_physical = (
        robust_global_shift_physical(
            previous_frame_positions,
            current_positions,
        )
    )

    assignment = assign_track_states(
        states=eligible_states,
        detections=detections,
        current_frame=current_frame,
        global_shift_physical=global_shift_physical,
    )

    rows = assignment["rows"]
    cols = assignment["cols"]

    matched_state_indices = set(
        rows.tolist()
    )

    matched_detection_indices = set(
        cols.tolist()
    )

    # --------------------------------------------------------
    # Continue matched tracks
    # --------------------------------------------------------

    for state_index, detection_index in zip(
        rows,
        cols,
    ):
        state = eligible_states[
            int(state_index)
        ]

        detection = detections.iloc[
            int(detection_index)
        ]

        previous_faces = "|".join(
            sorted(state["last_boundary_faces"])
        )

        previous_boundary_pending = bool(
            state["boundary_pending"]
        )

        was_reacquired, previous_boundary = (
            update_matched_state(
                state=state,
                detection=detection,
                current_frame=current_frame,
            )
        )

        is_boundary_related = bool(
            assignment["boundary_related"][
                state_index,
                detection_index,
            ]
        )

        if was_reacquired:
            match_type = "boundary_reacquired"
        elif is_boundary_related:
            match_type = "boundary_partial"
        else:
            match_type = "normal"

        distance_um = assignment[
            "distance_matrix"
        ][state_index, detection_index]

        cost = assignment["cost_matrix"][
            state_index,
            detection_index,
        ]

        append_track_record(
            track_id=state["track_id"],
            frame=current_frame,
            cell_index=int(
                detections.index[
                    detection_index
                ]
            ),
            detection=detection,
            match_type=match_type,
            match_distance_um=distance_um,
            match_cost=cost,
            template_reliable=state[
                "template_reliable"
            ],
        )

        current_boundary = bool(
            detection["touches_boundary"]
        )

        if was_reacquired:
            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": (
                        "boundary_reacquired"
                    ),
                    "boundary_faces": detection[
                        "boundary_faces"
                    ],
                    "missing_frames": 0,
                    "reacquired_frame": (
                        current_frame
                    ),
                    "confidence": float(
                        max(
                            0.0,
                            1.0 - min(cost, 1.0),
                        )
                    ),
                }
            )

        elif previous_boundary and not current_boundary:
            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": (
                        "entered_interior"
                    ),
                    "boundary_faces": (
                        previous_faces
                    ),
                    "missing_frames": 0,
                    "reacquired_frame": np.nan,
                    "confidence": float(
                        max(
                            0.0,
                            1.0 - min(cost, 1.0),
                        )
                    ),
                }
            )

        elif (
            not previous_boundary
            and current_boundary
        ):
            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": (
                        "boundary_exit_started"
                    ),
                    "boundary_faces": detection[
                        "boundary_faces"
                    ],
                    "missing_frames": 0,
                    "reacquired_frame": np.nan,
                    "confidence": float(
                        max(
                            0.0,
                            1.0 - min(cost, 1.0),
                        )
                    ),
                }
            )

    # --------------------------------------------------------
    # Preserve unmatched boundary tracks temporarily
    # --------------------------------------------------------

    for state_index, state in enumerate(
        eligible_states
    ):
        if state_index in matched_state_indices:
            continue

        last_was_boundary = bool(
            state["last_detection"].get(
                "touches_boundary",
                False,
            )
        )

        if (
            state["boundary_pending"]
            or last_was_boundary
        ):
            state["missed_frames"] += 1
            state["boundary_pending"] = True

            predicted_position = (
                predict_state_position(
                    state,
                    current_frame,
                    global_shift_physical,
                )
            )

            boundary_predictions.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "predicted_z": (
                        predicted_position[0]
                        / VOXEL_SIZE_ZYX[0]
                    ),
                    "predicted_y": (
                        predicted_position[1]
                        / VOXEL_SIZE_ZYX[1]
                    ),
                    "predicted_x": (
                        predicted_position[2]
                        / VOXEL_SIZE_ZYX[2]
                    ),
                    "missing_frames": state[
                        "missed_frames"
                    ],
                    "boundary_faces": "|".join(
                        sorted(
                            state[
                                "last_boundary_faces"
                            ]
                        )
                    ),
                }
            )

            boundary_events.append(
                {
                    "track_id": state["track_id"],
                    "frame": current_frame,
                    "event_type": (
                        "boundary_missing"
                    ),
                    "boundary_faces": "|".join(
                        sorted(
                            state[
                                "last_boundary_faces"
                            ]
                        )
                    ),
                    "missing_frames": state[
                        "missed_frames"
                    ],
                    "reacquired_frame": np.nan,
                    "confidence": np.nan,
                }
            )

            if (
                state["missed_frames"]
                > BOUNDARY_MAX_MISSING_FRAMES
            ):
                state["active"] = False

                boundary_events.append(
                    {
                        "track_id": (
                            state["track_id"]
                        ),
                        "frame": current_frame,
                        "event_type": (
                            "boundary_exit_confirmed"
                        ),
                        "boundary_faces": "|".join(
                            sorted(
                                state[
                                    "last_boundary_faces"
                                ]
                            )
                        ),
                        "missing_frames": state[
                            "missed_frames"
                        ],
                        "reacquired_frame": np.nan,
                        "confidence": np.nan,
                    }
                )

        else:
            state["active"] = False

    # --------------------------------------------------------
    # Create tracks only after pending boundary tracks were
    # allowed to compete for every current detection.
    # --------------------------------------------------------

    new_track_count = 0

    for detection_index in range(
        len(detections)
    ):
        if detection_index in matched_detection_indices:
            continue

        detection = detections.iloc[
            detection_index
        ]

        track_id = next_track_id
        next_track_id += 1
        new_track_count += 1

        state = make_track_state(
            track_id=track_id,
            frame=current_frame,
            detection=detection,
        )

        track_states[track_id] = state

        is_boundary = bool(
            detection["touches_boundary"]
        )

        match_type = (
            "boundary_entry"
            if is_boundary
            else "new_interior"
        )

        append_track_record(
            track_id=track_id,
            frame=current_frame,
            cell_index=int(
                detections.index[
                    detection_index
                ]
            ),
            detection=detection,
            match_type=match_type,
            match_distance_um=None,
            match_cost=None,
            template_reliable=state[
                "template_reliable"
            ],
        )

        if is_boundary:
            boundary_events.append(
                {
                    "track_id": track_id,
                    "frame": current_frame,
                    "event_type": (
                        "boundary_entry"
                    ),
                    "boundary_faces": detection[
                        "boundary_faces"
                    ],
                    "missing_frames": 0,
                    "reacquired_frame": np.nan,
                    "confidence": np.nan,
                }
            )

    print(
        f"{current_frame - 1:03d}"
        f"->{current_frame:03d}: "
        f"{len(rows):3d} matches | "
        f"{new_track_count:2d} new tracks | "
        f"{sum(state['active'] and state['boundary_pending'] for state in track_states.values()):2d} "
        "boundary-pending"
    )


tracks = pd.DataFrame(track_records)
boundary_events = pd.DataFrame(boundary_events)
boundary_predictions = pd.DataFrame(
    boundary_predictions
)

print()
print(f"Track records: {len(tracks):,}")
print(f"Unique tracks: {tracks['track_id'].nunique():,}")
print(f"Boundary events: {len(boundary_events):,}")
print(
    "Boundary reacquisitions:",
    (
        int(
            (
                boundary_events["event_type"]
                == "boundary_reacquired"
            ).sum()
        )
        if not boundary_events.empty
        else 0
    ),
)


000->001: 188 matches | 23 new tracks | 82 boundary-pending
001->002: 188 matches | 21 new tracks | 89 boundary-pending
002->003: 187 matches | 27 new tracks | 95 boundary-pending
003->004: 192 matches | 19 new tracks | 97 boundary-pending
004->005: 183 matches | 16 new tracks | 94 boundary-pending
005->006: 182 matches | 25 new tracks | 98 boundary-pending
006->007: 192 matches | 17 new tracks | 93 boundary-pending
007->008: 189 matches | 16 new tracks | 94 boundary-pending
008->009: 193 matches | 13 new tracks | 93 boundary-pending
009->010: 196 matches | 13 new tracks | 87 boundary-pending
010->011: 190 matches | 18 new tracks | 93 boundary-pending
011->012: 202 matches | 13 new tracks | 100 boundary-pending
012->013: 196 matches | 24 new tracks | 109 boundary-pending
013->014: 207 matches | 15 new tracks | 102 boundary-pending
014->015: 200 matches | 22 new tracks | 108 boundary-pending
015->016: 102 matches | 104 new tracks | 105 boundary-pending
016->017: 187 matches | 24 new tra

In [21]:
# ============================================================
# Tracking summary
# ============================================================

summary = {
    "track_records": len(tracks),
    "unique_tracks": tracks["track_id"].nunique(),
    "boundary_track_records": int(
        tracks["touches_boundary"].sum()
    ),
    "boundary_reacquisitions": int(
        (
            boundary_events["event_type"]
            == "boundary_reacquired"
        ).sum()
    ) if not boundary_events.empty else 0,
    "confirmed_boundary_exits": int(
        (
            boundary_events["event_type"]
            == "boundary_exit_confirmed"
        ).sum()
    ) if not boundary_events.empty else 0,
}

pd.Series(summary)


track_records               4223
unique_tracks                660
boundary_track_records      1523
boundary_reacquisitions       77
confirmed_boundary_exits     127
dtype: int64

## Save boundary-aware tracking results

`tracks.csv` contains observed detections only. Predicted positions used while
a boundary track is temporarily missing are stored separately in
`boundary_predictions.csv`.


In [22]:
# ============================================================
# Save boundary-aware tracking results
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

tracks.to_csv(
    OUTPUT_DIR / "tracks.csv",
    index=False,
)

boundary_events.to_csv(
    OUTPUT_DIR / "boundary_events.csv",
    index=False,
)

boundary_predictions.to_csv(
    OUTPUT_DIR / "boundary_predictions.csv",
    index=False,
)

boundary_counts.to_csv(
    OUTPUT_DIR / "boundary_detection_counts.csv",
    index=False,
)

final_track_states = pd.DataFrame(
    [
        {
            "track_id": state["track_id"],
            "active": state["active"],
            "last_frame": state["last_frame"],
            "missed_frames": state["missed_frames"],
            "boundary_pending": state["boundary_pending"],
            "last_boundary_faces": "|".join(
                sorted(state["last_boundary_faces"])
            ),
            "template_reliable": state[
                "template_reliable"
            ],
            "template_count": state[
                "template_count"
            ],
            "velocity_z_um_per_frame": float(
                state["velocity_physical"][0]
            ),
            "velocity_y_um_per_frame": float(
                state["velocity_physical"][1]
            ),
            "velocity_x_um_per_frame": float(
                state["velocity_physical"][2]
            ),
        }
        for state in track_states.values()
    ]
).sort_values("track_id")

final_track_states.to_csv(
    OUTPUT_DIR / "track_states.csv",
    index=False,
)

metadata = {
    "architecture": "boundary_aware_persistent_tracking",
    "sample_id": SAMPLE_ID,
    "volume_shape_zyx": (
        VOLUME_SHAPE_ZYX.tolist()
    ),
    "voxel_size_zyx": (
        VOXEL_SIZE_ZYX.tolist()
    ),
    "max_distance_um": MAX_DISTANCE_UM,
    "boundary_max_distance_um": (
        BOUNDARY_MAX_DISTANCE_UM
    ),
    "boundary_distance_per_missing_frame_um": (
        BOUNDARY_DISTANCE_PER_MISSING_FRAME_UM
    ),
    "global_shift_max_pair_distance_um": (
        GLOBAL_SHIFT_MAX_PAIR_DISTANCE_UM
    ),
    "max_volume_ratio_interior": (
        MAX_VOLUME_RATIO_INTERIOR
    ),
    "max_volume_ratio_boundary": (
        MAX_VOLUME_RATIO_BOUNDARY
    ),
    "boundary_max_missing_frames": (
        BOUNDARY_MAX_MISSING_FRAMES
    ),
    "boundary_margin_um": BOUNDARY_MARGIN_UM,
    "interior_weights": {
        "distance": INTERIOR_W_DISTANCE,
        "size": INTERIOR_W_SIZE,
        "shape": INTERIOR_W_SHAPE,
        "intensity": INTERIOR_W_INTENSITY,
        "bbox": INTERIOR_W_BBOX,
    },
    "boundary_weights": {
        "distance": BOUNDARY_W_DISTANCE,
        "motion": BOUNDARY_W_MOTION,
        "intensity": BOUNDARY_W_INTENSITY,
        "face": BOUNDARY_W_FACE,
    },
    "track_records": int(len(tracks)),
    "unique_tracks": int(
        tracks["track_id"].nunique()
    ),
    "boundary_events": int(
        len(boundary_events)
    ),
    "boundary_reacquisitions": int(
        (
            boundary_events["event_type"]
            == "boundary_reacquired"
        ).sum()
    ) if not boundary_events.empty else 0,
}

with open(
    OUTPUT_DIR / "metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=4,
    )

print(f"Saved {len(tracks):,} track records.")
print(f"Output directory: {OUTPUT_DIR}")
print()
print("Files:")
for output_file in sorted(OUTPUT_DIR.iterdir()):
    print(" ", output_file.name)


Saved 4,223 track records.
Output directory: D:\Projects\Kaggle\cell-tracking\data\sample\processed\stage_7_cell_tracking

Files:
  boundary_detection_counts.csv
  boundary_events.csv
  boundary_predictions.csv
  metadata.json
  track_states.csv
  tracks.csv
